# 01 — RAG Pipeline Exploration

**FreightIQ** uses a Retrieval-Augmented Generation (RAG) pipeline to give the LLM access to historical supply chain disruption data when generating risk assessments and recommendations.

This notebook walks through every layer of the pipeline interactively:

1. **Embeddings** — How we convert text into vectors using Ollama's `nomic-embed-text`
2. **Vector Store** — Storing and querying documents in ChromaDB
3. **Retriever** — The high-level retrieval interface used by the agent
4. **Prompt Templates** — How retrieved context is injected into LLM prompts
5. **End-to-End Query** — Putting it all together

---

### Prerequisites

Before running this notebook, ensure:
- Ollama is running locally (`ollama serve`)
- Models are pulled: `ollama pull nomic-embed-text` and `ollama pull qwen2.5:7b-instruct`
- Synthetic data has been generated: `python scripts/generate_data.py`

In [ ]:
# Ensure the project root is on sys.path so `src.*` imports work
import sys, os
sys.path.insert(0, os.path.abspath(".."))

# Load environment variables from .env
from dotenv import load_dotenv
load_dotenv("../.env")

print("✅ Project root added to path")

---
## 1. Embeddings

FreightIQ uses **Ollama's `nomic-embed-text`** model (~274 MB) to create 768-dimensional embeddings entirely on your local machine — no data leaves your network.

The embedding function is centralised in `src/llm/llm_client.py` and exposed via `src/rag/embeddings.py`.

In [ ]:
from src.llm.llm_client import get_embeddings, check_ollama_health

# First, verify Ollama is reachable
print(f"Ollama healthy: {check_ollama_health()}")

In [ ]:
# Create the embedding function
embed_fn = get_embeddings()
print(f"Embedding model: {embed_fn.model}")
print(f"Base URL: {embed_fn.base_url}")

In [ ]:
# Embed a sample supply chain sentence
sample_texts = [
    "Port of Rotterdam workers declared an indefinite strike.",
    "Typhoon Gaemi forces closure of Kaohsiung port in Taiwan.",
    "Container shipping rates from Shanghai to Los Angeles increased 40%.",
]

vectors = embed_fn.embed_documents(sample_texts)

print(f"Number of vectors: {len(vectors)}")
print(f"Vector dimension:  {len(vectors[0])}")
print(f"First 8 values:    {vectors[0][:8]}")

In [ ]:
# Compute cosine similarity between the embeddings to verify
# semantically related texts are closer together
import numpy as np

def cosine_sim(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("Cosine similarities:")
print(f"  Strike vs Typhoon (both port disruptions):   {cosine_sim(vectors[0], vectors[1]):.4f}")
print(f"  Strike vs Shipping rates (less related):     {cosine_sim(vectors[0], vectors[2]):.4f}")
print(f"  Typhoon vs Shipping rates:                   {cosine_sim(vectors[1], vectors[2]):.4f}")
print()
print("📌 Higher similarity = more semantically related.")

---
## 2. Vector Store (ChromaDB)

Documents are stored in **ChromaDB** — a lightweight, persistent vector database.

The `DisruptionVectorStore` class in `src/rag/vector_store.py` handles:
- Creating/connecting to the ChromaDB collection
- Upserting documents with metadata
- Similarity search with optional metadata filters

Let's load the synthetic disruption data and ingest it.

In [ ]:
import json

# Load the synthetic disruption reports
with open("../data/synthetic/disruptions.json", "r") as f:
    disruptions = json.load(f)

print(f"Loaded {len(disruptions)} disruption records")
print(f"\nSample record:")
print(json.dumps(disruptions[0], indent=2))

In [ ]:
from src.rag.vector_store import DisruptionVectorStore

# Initialize the vector store (creates/connects to ChromaDB)
vstore = DisruptionVectorStore()
print(f"Collection: {vstore.collection_name}")
print(f"Persist dir: {vstore.persist_dir}")

In [ ]:
# Prepare documents for upsert
# The upsert_documents method expects dicts with 'title', 'description', and metadata fields
docs_for_upsert = []
for d in disruptions:
    docs_for_upsert.append({
        "id": d["id"],
        "title": f"{d['event_type']} at {d['port']}",
        "description": d["description"],
        "region": d["region"],
        "event_type": d["event_type"],
        "date": d["date"],
        "source": d["source"],
    })

# Upsert into ChromaDB
vstore.upsert_documents(docs_for_upsert)

In [ ]:
# Raw similarity search — find docs most related to a query
results = vstore.similarity_search(
    query="port strike affecting container shipping in Europe",
    k=3
)

print(f"Found {len(results)} results\n")
for i, doc in enumerate(results, 1):
    print(f"--- Result {i} ---")
    print(f"  Region:     {doc.metadata.get('region', 'N/A')}")
    print(f"  Event type: {doc.metadata.get('event_type', 'N/A')}")
    print(f"  Date:       {doc.metadata.get('date', 'N/A')}")
    print(f"  Content:    {doc.page_content[:150]}...")
    print()

In [ ]:
# Filtered search — only return results from a specific region
filtered_results = vstore.similarity_search(
    query="shipping delays",
    k=3,
    filter_dict={"region": "Asia"}
)

print(f"Found {len(filtered_results)} results (filtered to Asia)\n")
for i, doc in enumerate(filtered_results, 1):
    print(f"--- Result {i} ---")
    print(f"  Region:     {doc.metadata.get('region')}")
    print(f"  Event type: {doc.metadata.get('event_type')}")
    print(f"  Content:    {doc.page_content[:150]}...")
    print()

---
## 3. Retriever

The `DisruptionRetriever` in `src/rag/retriever.py` wraps the vector store with a cleaner API and adds a `format_docs()` method that converts results into a formatted string ready for injection into LLM prompts.

In [ ]:
from src.rag.retriever import DisruptionRetriever

retriever = DisruptionRetriever()

# Query with metadata filters — exactly as the LangGraph agent does
results = retriever.query(
    query="port strike affecting container shipping",
    filters={"event_type": "labour_dispute"},
    k=3
)

print(f"Retrieved {len(results)} documents\n")
for doc in results:
    print(f"  [{doc.metadata.get('event_type')}] {doc.metadata.get('region')} — {doc.page_content[:100]}...")

In [ ]:
# Format the results for LLM context injection
formatted = retriever.format_docs(results)

print("Formatted context string (this is what goes into the LLM prompt):")
print("=" * 70)
print(formatted[:1000])
print("=" * 70)

---
## 4. Prompt Templates

FreightIQ uses structured prompt templates in `src/rag/prompts.py` to guide the LLM. Let's examine how the retrieved context gets injected.

In [ ]:
from src.rag.prompts import SYSTEM_PROMPT, DETECT_PROMPT, RECOMMEND_PROMPT, QUERY_PROMPT

print("=== SYSTEM PROMPT ===")
print(SYSTEM_PROMPT)
print()
print("=== QUERY PROMPT (template) ===")
print(QUERY_PROMPT)

In [ ]:
# Demonstrate how the QUERY_PROMPT gets filled with retrieved context
user_question = "Which European ports are most affected by labour disputes?"

# Step 1: Retrieve relevant documents
docs = retriever.query(query=user_question, k=3)
context = retriever.format_docs(docs)

# Step 2: Fill the template
filled_prompt = QUERY_PROMPT.format(context=context, question=user_question)

print("Filled prompt (first 800 chars):")
print("=" * 70)
print(filled_prompt[:800])
print("...")
print("=" * 70)

---
## 5. End-to-End: RAG Query via the LLM

Now let's put it all together: retrieve context → inject into prompt → send to the local Ollama LLM → get an answer.

> **Note:** This cell requires Ollama to be running with `qwen2.5:7b-instruct` pulled.

In [ ]:
from src.llm.llm_client import get_llm

llm = get_llm(temperature=0.0)
print(f"LLM model: {llm.model}")

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

def rag_query(question: str, k: int = 5) -> str:
    """Complete RAG pipeline: retrieve → prompt → LLM → answer."""
    # Retrieve
    docs = retriever.query(query=question, k=k)
    context = retriever.format_docs(docs)
    
    # Build prompt
    filled = QUERY_PROMPT.format(context=context, question=question)
    
    # Call LLM
    response = llm.invoke([
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=filled),
    ])
    
    return response.content


# Ask a question
answer = rag_query("What types of disruptions have affected shipping in Europe recently?")
print(answer)

In [ ]:
# Try another question
answer2 = rag_query("What is the recommended mitigation when a port is congested?")
print(answer2)

---
## 6. Experiment: Chunk Size & Retrieval Quality

A key RAG design decision is how many documents (`k`) to retrieve. Too few = missing context. Too many = noisy context that confuses the LLM.

Let's compare retrieval quality at different `k` values.

In [ ]:
test_query = "typhoon causing port closure"

for k in [1, 3, 5, 10]:
    docs = retriever.query(query=test_query, k=k)
    
    # Count how many are actually weather events (relevant)
    relevant = sum(1 for d in docs if d.metadata.get("event_type") == "weather_event")
    
    print(f"k={k:2d} → {len(docs)} docs retrieved, {relevant} are weather_event (precision: {relevant/len(docs)*100:.0f}%)")

In [ ]:
# Compare: filtered vs unfiltered retrieval
print("=== UNFILTERED ===")
unfiltered = retriever.query(query="shipping delays", k=5)
for d in unfiltered:
    print(f"  {d.metadata.get('event_type'):20s} | {d.metadata.get('region')}")

print("\n=== FILTERED (event_type = weather_event) ===")
filtered = retriever.query(query="shipping delays", filters={"event_type": "weather_event"}, k=5)
for d in filtered:
    print(f"  {d.metadata.get('event_type'):20s} | {d.metadata.get('region')}")

print("\n📌 Metadata filters are crucial for precision in domain-specific RAG.")

---
## Key Takeaways

| Design Decision | Choice | Why |
|---|---|---|
| Embedding model | `nomic-embed-text` (Ollama) | Runs locally, no API costs, 768-dim vectors |
| Vector store | ChromaDB (persistent) | Lightweight, embedded, no server needed |
| Default `k` | 5 | Good balance of precision and recall for our corpus size |
| Metadata filters | Region, event_type, source | Enables targeted retrieval for specific agent nodes |
| Prompt structure | System + filled template | Keeps LLM responses focused and structured |

### Next Steps

- See `02_xgboost_training.ipynb` for the ML risk scoring model
- See `src/agent/nodes.py` for how the agent's Retrieve node uses this pipeline